<a href="https://colab.research.google.com/github/Viswr/samples/blob/master/FakeNewsDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# INSTALL REQUIRED LIBRARIES
# =====================================================

!pip install transformers datasets torch scikit-learn

In [ ]:
# =====================================================
# IMPORT LIBRARIES
# =====================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [ ]:
# =====================================================
# CREATE LARGER SYNTHETIC DATASET
# =====================================================

fake_news = [
"Scientists confirm aliens living on Mars",
"Government secretly controlling weather",
"Celebrity discovers immortality pill",
"Secret society running world politics",
"Ancient pyramid found on the moon",
"Scientists discover teleportation machine",
"Aliens communicating with world leaders",
"Hidden cure for aging discovered",
"Time traveler reveals future events",
"New evidence proves earth is hollow"
]

real_news = [
"NASA launches new climate monitoring satellite",
"Researchers publish new study on climate change",
"Central bank announces new interest rate policy",
"Scientists develop new cancer treatment",
"Government releases annual economic report",
"University researchers discover new protein",
"Tech company announces new AI chip",
"Medical researchers publish vaccine results",
"Space agency plans new Mars mission",
"International summit discusses climate policy"
]

texts = fake_news + real_news

labels = [1]*len(fake_news) + [0]*len(real_news)

df = pd.DataFrame({
"text":texts,
"label":labels
})

df

,text,label
0,Scientists confirm aliens living on Mars,1
1,Government secretly controlling weather,1
2,Celebrity discovers immortality pill,1
3,Secret society running world politics,1
4,Ancient pyramid found on the moon,1
5,Scientists discover teleportation machine,1
6,Aliens communicating with world leaders,1
7,Hidden cure for aging discovered,1
8,Time traveler reveals future events,1
9,New evidence proves earth is hollow,1


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
df["text"],
df["label"],
test_size=0.3,
random_state=42
)

In [ ]:
# =====================================================
# TOKENIZER
# =====================================================

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# =====================================================
# TOKENIZE DATA
# =====================================================

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True
)

In [ ]:
# =====================================================
# DATASET CLASS
# =====================================================

class NewsDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        item["labels"] = torch.tensor(self.labels.iloc[idx])

        return item

    def __len__(self):
        return len(self.labels)


train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

In [ ]:
# =====================================================
# LOAD TRANSFORMER MODEL
# =====================================================

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# =====================================================
# TRAINING CONFIG
# =====================================================

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=20,
    eval_strategy="epoch",
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# =====================================================
# TRAINER
# =====================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
# =====================================================
# TRAIN MODEL
# =====================================================

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.707217
2,No log,0.671112
3,No log,0.636029
4,No log,0.564499
5,No log,0.500777
6,No log,0.438018
7,No log,0.386570
8,No log,0.361507
9,No log,0.356395
10,No log,0.372013


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=80, training_loss=0.23329453468322753, metrics={'train_runtime': 63.5325, 'train_samples_per_second': 4.407, 'train_steps_per_second': 1.259, 'total_flos': 651987977760.0, 'train_loss': 0.23329453468322753, 'epoch': 20.0})

In [ ]:
# =====================================================
# EVALUATION
# =====================================================

predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, preds))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       1.00      0.75      0.86         4

    accuracy                           0.83         6
   macro avg       0.83      0.88      0.83         6
weighted avg       0.89      0.83      0.84         6



In [ ]:
test_news = [
"Scientists discover new alien civilization",
"Secret government project controls weather",
"Researchers develop new vaccine technology",
"NASA launches satellite for space research",
"Scientists publish study on climate change"
]

In [ ]:
for t in test_news:
    print(t, "→", predict_news(t))

NameError: name 'predict_news' is not defined